# FX Statistical Arbitrage — Cointegration Pairs Trading

Find a cointegrated pair among G10 USD FX rates and trade mean-reversion of the spread. A research backtest on **real data**.

- Engle-Granger cointegration test on every pair; trade the most cointegrated.
- Rolling hedge ratio (rolling OLS) and rolling z-score of the spread.
- Enter when |z| > 2 (fade the deviation), exit when |z| < 0.5. Signals lagged one day; transaction costs; in-sample / out-of-sample split.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from statsmodels.tsa.stattools import coint

TICKERS = {"EURUSD=X":"EUR","GBPUSD=X":"GBP","AUDUSD=X":"AUD","NZDUSD=X":"NZD",
           "USDCAD=X":"CAD","USDCHF=X":"CHF","USDNOK=X":"NOK","USDSEK=X":"SEK"}
START="2010-01-01"; LOOKBACK=60; Z_ENTRY=2.0; Z_EXIT=0.5
COST_BPS=2.0; OOS_SPLIT="2019-01-01"; TRADING_DAYS=252

In [3]:
%pip install statsmodels yfinance


Note: you may need to restart the kernel to use updated packages.


## 1. Load real G10 FX prices (log)

In [5]:
import yfinance as yf
px = yf.download(list(TICKERS), start=START, progress=False)["Close"]
px = px.rename(columns=TICKERS).ffill().dropna()
logpx = np.log(px)
print(f"{len(logpx)} days, {logpx.shape[1]} pairs")
logpx.tail()

4283 days, 8 pairs


Ticker,AUD,EUR,GBP,NZD,CAD,CHF,NOK,SEK
Date,,,,,,,,
2026-06-09,-0.350906,0.142232,0.287682,-0.544108,0.333346,-0.225434,2.248145,2.244970
2026-06-10,-0.353378,0.142832,0.290593,-0.543434,0.333167,-0.224069,2.252349,2.249500
2026-06-11,-0.357504,0.142855,0.289831,-0.545708,0.332665,-0.223206,2.247796,2.252979
2026-06-12,-0.349855,0.146067,0.293217,-0.539086,0.334499,-0.229350,2.252070,2.244613
2026-06-13,-0.349741,0.146067,0.293217,-0.538655,0.335686,-0.227654,2.252070,2.244613


## 2. Find the most cointegrated pair (Engle-Granger)

In [ ]:
best = None
for a, b in combinations(logpx.columns, 2):
    _, pval, _ = coint(logpx[a], logpx[b])
    if best is None or pval < best[2]:
        best = (a, b, pval)
a, b, pval = best
print(f"Most cointegrated pair: {a}/{b}  (Engle-Granger p-value = {pval:.4f})")
if pval > 0.05:
    print("Note: p > 0.05 — no strong cointegration; read results with caution.")

## 3. Rolling spread, z-score, and mean-reversion signals

In [ ]:
y, x = logpx[a], logpx[b]
beta = y.rolling(LOOKBACK).cov(x) / x.rolling(LOOKBACK).var()   # rolling hedge ratio
spread = y - beta*x
z = (spread - spread.rolling(LOOKBACK).mean()) / spread.rolling(LOOKBACK).std()

pos = np.zeros(len(z)); cur = 0.0; zv = z.to_numpy()
for t in range(len(zv)):
    if np.isnan(zv[t]):
        pos[t] = cur; continue
    if cur == 0:
        if zv[t] > Z_ENTRY:   cur = -1.0      # spread too high -> short it
        elif zv[t] < -Z_ENTRY: cur = 1.0      # spread too low  -> long it
    elif abs(zv[t]) < Z_EXIT:
        cur = 0.0
    pos[t] = cur
pos = pd.Series(pos, index=z.index).shift(1).fillna(0)         # lag one day

## 4. Backtest and performance

In [ ]:
dspread = spread.diff()
gross = pos*dspread
trades = pos.diff().abs()
net = (gross - trades.fillna(0)*(COST_BPS/1e4)).dropna()

def performance(r):
    ar = r.mean()*TRADING_DAYS; av = r.std()*np.sqrt(TRADING_DAYS)
    curve = (1+r).cumprod(); dd = (curve/curve.cummax()-1).min()
    return {"Annual return":ar,"Annual volatility":av,
            "Sharpe ratio":ar/av if av else np.nan,
            "Max drawdown":dd,"Hit rate":(r>0).mean()}
def show(title, m):
    print(f"\n{title}\n"+"-"*len(title))
    for k,v in m.items():
        pct = any(w in k.lower() for w in ("return","volatility","drawdown","rate"))
        print(f"  {k:<20}{v:>8.1%}" if pct else f"  {k:<20}{v:>8.2f}")

show("Full sample - net", performance(net))
show(f"In-sample (before {OOS_SPLIT})",  performance(net[net.index<OOS_SPLIT]))
show(f"Out-of-sample (from {OOS_SPLIT})", performance(net[net.index>=OOS_SPLIT]))
print(f"\nRound-trips (approx): {int(pos.diff().abs().gt(0).sum()/2)}")

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(11,7),sharex=True)
ax[0].plot((1+net).cumprod().index, (1+net).cumprod())
ax[0].set_title(f"Pairs trade {a}/{b} - cumulative net PnL (log-spread units)")
ax[0].set_ylabel("Growth of 1")
ax[1].plot(z.index, z, lw=0.8, color="grey")
ax[1].axhline(Z_ENTRY, ls="--", c="r", lw=0.8); ax[1].axhline(-Z_ENTRY, ls="--", c="r", lw=0.8)
ax[1].axhline(0, c="k", lw=0.5); ax[1].set_ylabel("Spread z-score")
fig.tight_layout(); fig.savefig("fx_statarb_results.png", dpi=120); plt.show()

## Limitations
- PnL is in log-spread units with a time-varying hedge ratio — a standard educational simplification, not an exact tradable return.
- Cointegration found in-sample can break out-of-sample, which is exactly why the OOS split matters.
- G10 USD pairs share the USD leg, so apparent cointegration can be partly common USD moves.